In [12]:
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

In [13]:
CSV_PATH = Path("cov_types.csv")
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"
RANDOM_STATE = 42
SAMPLE_SIZE = 50_000

In [14]:
COLUNAS = (
    [
        "Elevation",
        "Aspect",
        "Slope",
        "Horizontal_Distance_To_Hydrology",
        "Vertical_Distance_To_Hydrology",
        "Horizontal_Distance_To_Roadways",
        "Hillshade_9am",
        "Hillshade_Noon",
        "Hillshade_3pm",
        "Horizontal_Distance_To_Fire_Points",
    ]
    + [f"Wilderness_Area{i}" for i in range(1, 5)]
    + [f"Soil_Type{i}" for i in range(1, 41)]
    + ["Cover_Type"]
)

In [19]:
def carregar_base() -> pd.DataFrame:
    """Carrega cov_types.csv ou baixa a base se ela ainda nao existir."""
    if CSV_PATH.exists():
        print(f"Lendo arquivo local: {CSV_PATH.resolve()}")
        return pd.read_csv(CSV_PATH)

    print("Arquivo cov_types.csv nao encontrado.")
    print("Baixando a base oficial Covertype da UCI...")

    try:
        df = pd.read_csv(UCI_URL, header=None, names=COLUNAS, compression="gzip")
    except Exception as erro:
        raise RuntimeError(
            "Nao consegui baixar a base. Verifique sua internet ou baixe "
            "covtype.data.gz da UCI manualmente."
        ) from erro

    df.to_csv(CSV_PATH, index=False)
    print(f"Base salva em: {CSV_PATH.resolve()}")
    return df

In [25]:
def mostrar_distribuicao(y:pd.Series) -> None:
    """Mostra contagem e porcentagem de cada classe."""
    contagem = y.value_counts().sort_index()
    percentual = (y.value_counts(normalize=True).sort_index() * 100).round(2)

    tabela = pd.DataFrame({
        "Contagem": contagem,
        "Percentual (%)": percentual
    })
    print("\nDistribuição da variável alvo:\n")
    print(tabela)

    classe_majoritaria = contagem.idxmax()
    classe_minoritaria = contagem.idxmin()

    print("\nClasse majoritária:", classe_majoritaria)
    print("Classe minoritária:", classe_minoritaria)

In [26]:
def avaliar(nome:str, modelo, X_test, y_test) -> None:
    """Avalia o modelo e imprime métricas."""
    pred = modelo.predict(X_test)

    print('\n' + '='*80)
    print(nome)
    print('='*80)

    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, pred), 4))

    print("\nMatriz de Confusão:")
    print(confusion_matrix(y_test, pred))

    print("\nRelatório de Classificação:")
    print(classification_report(y_test, pred))

In [27]:
def main() -> None:
    df = carregar_base()
    
    print('\nFormato original da base:', df.shape)
    print('\nPrimeiras linhas:')
    print(df.head())

    # Usamos uma amostra para o treino ficar rapido no notebook/computador local.
    # Em um projeto real, voce usaria a base completa.
    if len(df) > SAMPLE_SIZE:
        df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
        print(f"\nUsando amostra de {SAMPLE_SIZE} linhas para treino.")

    X = df.drop(columns="Cover_Type")
    y = df["Cover_Type"]
    
    mostrar_distribuicao(y)
    
    # stratify=y garante que a proporção das classes seja mantida no treino e teste.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # DummyClassifier é um modelo de baseline que sempre prevê a classe majoritária.
    # Aqui ele sempre prediz a classe mais frequente do conjunto de treino.
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)
    avaliar("Modelo bobo: sempre prediz a classe majoritária", dummy, X_test, y_test)

    # Este é o primeiro modelo real, sem pré-processamento. Ele pode ter um desempenho ruim, mas é um ponto de partida.
    baseline = RandomForestClassifier(n_estimators=80,
                                        random_state=RANDOM_STATE,
                                        n_jobs=-1)
    baseline.fit(X_train, y_train)
    avaliar("Baseline: Random Forest sem pré-processamento", baseline, X_test, y_test)

In [28]:
if __name__ == "__main__":
    main()

Lendo arquivo local: C:\Users\Olive\VS Code\Python\meu-projeto\machine_learning_and_Data_Science\9_dados_desbalanceados\cov_types.csv

Formato original da base: (581012, 55)

Primeiras linhas:
   Elevation  Aspect  Slope  Horizontal_Distance_To_Hydrology  \
0       2596      51      3                               258   
1       2590      56      2                               212   
2       2804     139      9                               268   
3       2785     155     18                               242   
4       2595      45      2                               153   

   Vertical_Distance_To_Hydrology  Horizontal_Distance_To_Roadways  \
0                               0                              510   
1                              -6                              390   
2                              65                             3180   
3                             118                             3090   
4                              -1                              391

c:\Users\Olive\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Olive\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Olive\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Baseline: Random Forest sem pré-processamento
Accuracy: 0.8777
Balanced Accuracy: 0.7435

Matriz de Confusão:
[[3193  463    0    0    1    0   23]
 [ 349 4467   35    0    1    7    1]
 [   0   37  556    0    0   24    0]
 [   0    0   15   29    0    0    0]
 [   2   84    4    0   67    2    0]
 [   1   36   74    1    0  181    0]
 [  57    6    0    0    0    0  284]]

Relatório de Classificação:
              precision    recall  f1-score   support

           1       0.89      0.87      0.88      3680
           2       0.88      0.92      0.90      4860
           3       0.81      0.90      0.85       617
           4       0.97      0.66      0.78        44
           5       0.97      0.42      0.59       159
           6       0.85      0.62      0.71       293
           7       0.92      0.82      0.87       347

    accuracy                           0.88     10000
   macro avg       0.90      0.74      0.80     10000
weighted avg       0.88      0.88      0.88     100